# 02  Enriquecimiento externo

**Proyecto:** Siniestros fatales de tr?nsito en el Per? 2021-2025  
**Curso:** Agentes Inteligentes | Grupo 5

Este notebook toma el dataset limpio generado en el notebook 01 (`siniestros_limpio.csv`) y lo enriquece con fuentes externas reales:

1. **MTC Red Vial Nacional 2022-2025**, cruzada por `COD CARRETERA`.
2. **IDH PNUD/IPE 2019**, cruzado por `DEPARTAMENTO + PROVINCIA + DISTRITO` cuando es posible.
3. **MTC Infraestructura vial por departamento 2025**, cruzada por `DEPARTAMENTO`.

El resultado se exporta a `data/procesada/siniestros_enriquecido.csv` junto con una tabla de trazabilidad.

## 0. Configuración

El notebook est? pensado para ejecutarse desde la ra?z del repositorio. No usa rutas absolutas ni tablas embebidas: todas las fuentes externas se leen desde `data/externas/`.

In [97]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_EXT = ROOT / "data" / "externas"
DATA_PROC = ROOT / "data" / "procesada"
DATA_PROC.mkdir(parents=True, exist_ok=True)

PATHS = {
    "siniestros_proc": DATA_PROC / "siniestros_limpio.csv",
    "mtc_rvn": DATA_EXT / "1_Red_vial_nacional_2022-2025.csv",
    "mtc_diccionario": DATA_EXT / "1_Diccionario_Datos_Red_vial_nacional_2022-2025.xlsx",
    "idh": DATA_EXT / "IDH-y-Componentes-2003-2019.xlsx",
    "mtc_departamento": DATA_EXT / "344790-red-vial-existente-del-sistema-nacional-de-carreteras-segun-departamento-2010-jun-2025.xlsx",
    "salida_enriquecido": DATA_PROC / "siniestros_enriquecido.csv",
    "salida_trazabilidad": DATA_PROC / "trazabilidad_variables_externas.csv",
}

print(f"ROOT detectado: {ROOT}")
print(f"DATA_EXT: {DATA_EXT}")
print(f"DATA_PROC: {DATA_PROC}")
for nombre, ruta in PATHS.items():
    if nombre.startswith("salida"):
        continue
    print(f"{nombre:18s}: {'OK' if ruta.exists() else 'NO ENCONTRADO'} -> {ruta}")

ROOT detectado: /home/node-vault07/projects/siniestros-fatales-peru-ml
DATA_EXT: /home/node-vault07/projects/siniestros-fatales-peru-ml/data/externas
DATA_PROC: /home/node-vault07/projects/siniestros-fatales-peru-ml/data/procesada
siniestros_proc   : OK -> /home/node-vault07/projects/siniestros-fatales-peru-ml/data/procesada/siniestros_limpio.csv
mtc_rvn           : OK -> /home/node-vault07/projects/siniestros-fatales-peru-ml/data/externas/1_Red_vial_nacional_2022-2025.csv
mtc_diccionario   : OK -> /home/node-vault07/projects/siniestros-fatales-peru-ml/data/externas/1_Diccionario_Datos_Red_vial_nacional_2022-2025.xlsx
idh               : OK -> /home/node-vault07/projects/siniestros-fatales-peru-ml/data/externas/IDH-y-Componentes-2003-2019.xlsx
mtc_departamento  : OK -> /home/node-vault07/projects/siniestros-fatales-peru-ml/data/externas/344790-red-vial-existente-del-sistema-nacional-de-carreteras-segun-departamento-2010-jun-2025.xlsx


In [98]:
def normalizar_texto(valor):
    """Normaliza texto para cruces: may?sculas, sin tildes y sin espacios m?ltiples."""
    if pd.isna(valor):
        return ""
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"\s+", " ", texto)
    return texto


def normalizar_codigo_ruta(valor):
    """Normaliza c?digos de carretera conservando guiones."""
    texto = normalizar_texto(valor)
    texto = texto.replace(" ", "")
    return texto


def moda_ponderada_por_longitud(grupo, col_valor, col_peso="LONGITUD"):
    """Devuelve la categor?a con mayor longitud acumulada dentro de una ruta."""
    tmp = grupo[[col_valor, col_peso]].dropna(subset=[col_valor]).copy()
    if tmp.empty:
        return np.nan
    tmp[col_peso] = pd.to_numeric(tmp[col_peso], errors="coerce").fillna(0)
    pesos = tmp.groupby(col_valor)[col_peso].sum().sort_values(ascending=False)
    if pesos.empty or pesos.iloc[0] == 0:
        modo = tmp[col_valor].mode()
        return modo.iloc[0] if not modo.empty else np.nan
    return pesos.index[0]


def validar_merge(df_antes, df_despues, nombre_fuente, cols_nuevas):
    """Valida integridad del merge y reporta cobertura."""
    print("=" * 72)
    print(f"VALIDACI?N POST-MERGE: {nombre_fuente}")
    print("=" * 72)
    print(f"Filas antes:   {len(df_antes):,}")
    print(f"Filas despu?s: {len(df_despues):,}")
    assert len(df_antes) == len(df_despues), "El merge cambi? la cantidad de filas. Revisar many-to-many."

    dist_antes = df_antes["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()
    dist_despues = df_despues["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()
    assert dist_antes == dist_despues, "Cambi? la distribuci?n de CATEGORIA_SEVERIDAD."
    print("Variable objetivo: inalterada")

    disponibles = [c for c in cols_nuevas if c in df_despues.columns]
    cobertura = df_despues[disponibles].notna().any(axis=1).mean() * 100 if disponibles else 0
    cruzados = int(df_despues[disponibles].notna().any(axis=1).sum()) if disponibles else 0
    no_cruzados = len(df_despues) - cruzados
    print(f"Cobertura fuente: {cobertura:.2f}% ({cruzados:,} cruzados; {no_cruzados:,} no cruzados)")
    print("Columnas nuevas:", disponibles)
    return cobertura, cruzados, no_cruzados


def agregar_trazabilidad(registros, variables, fuente, archivo_origen, url, llave_cruce, anio_dato, nivel_geografico, cobertura, cruzados, no_cruzados, observaciones):
    for variable in variables:
        registros.append({
            "variable": variable,
            "fuente": fuente,
            "archivo_origen": archivo_origen,
            "url_fuente": url,
            "llave_cruce": llave_cruce,
            "anio_dato": anio_dato,
            "nivel_geografico": nivel_geografico,
            "cobertura_pct": round(float(cobertura), 2),
            "registros_cruzados": int(cruzados),
            "registros_no_cruzados": int(no_cruzados),
            "observaciones": observaciones,
        })

## 1. Carga del dataset limpio

Se carga `siniestros_limpio.csv` desde la ra?z del repositorio. Si no existe, se intenta leer desde `data/procesada/`. Esta segunda opci?n se deja para futuras reorganizaciones del repositorio.

In [99]:
# Preferir la versión canónica en data/procesada; mantener fallback por compatibilidad
if PATHS["siniestros_proc"].exists():
    path_siniestros = PATHS["siniestros_proc"]
elif PATHS["siniestros_raiz"].exists():
    path_siniestros = PATHS["siniestros_raiz"]
else:
    raise FileNotFoundError("No se encontró siniestros_limpio.csv en data/procesada/ ni en la raíz del repo.")

try:
    df = pd.read_csv(path_siniestros, encoding="utf-8-sig")
except UnicodeDecodeError:
    df = pd.read_csv(path_siniestros, encoding="latin1")

df_original = df.copy()
filas_originales = len(df)
dist_objetivo_original = df["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict()

print(f"Archivo base: {path_siniestros}")
print(f"Dataset limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(df[["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "COD CARRETERA", "CATEGORIA_SEVERIDAD"]].head())

Archivo base: /home/node-vault07/projects/siniestros-fatales-peru-ml/data/procesada/siniestros_limpio.csv
Dataset limpio: 9,106 filas x 36 columnas
  DEPARTAMENTO PROVINCIA                 DISTRITO   COD CARRETERA  \
0         LIMA    HUARAL                   HUARAL          LM-671   
1         LIMA      LIMA               PACHACAMAC  NO CORRESPONDE   
2  LA LIBERTAD      VIRU                     VIRU         LI-1150   
3  LA LIBERTAD      VIRU                     VIRU           PE-1N   
4         LIMA      LIMA  VILLA MARIA DEL TRIUNFO  NO CORRESPONDE   

  CATEGORIA_SEVERIDAD  
0                LEVE  
1    GRAVE_LESIONADOS  
2    GRAVE_MORTALIDAD  
3                LEVE  
4                LEVE  


In [100]:
# Llaves normalizadas auxiliares. Se eliminar?n antes de exportar.
df["_DEP_NORM"] = df["DEPARTAMENTO"].apply(normalizar_texto)
df["_PROV_NORM"] = df["PROVINCIA"].apply(normalizar_texto)
df["_DIST_NORM"] = df["DISTRITO"].apply(normalizar_texto)
df["_COD_CARRETERA_NORM"] = df["COD CARRETERA"].apply(normalizar_codigo_ruta)

print(df[["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "COD CARRETERA", "_COD_CARRETERA_NORM"]].head())

  DEPARTAMENTO PROVINCIA                 DISTRITO   COD CARRETERA  \
0         LIMA    HUARAL                   HUARAL          LM-671   
1         LIMA      LIMA               PACHACAMAC  NO CORRESPONDE   
2  LA LIBERTAD      VIRU                     VIRU         LI-1150   
3  LA LIBERTAD      VIRU                     VIRU           PE-1N   
4         LIMA      LIMA  VILLA MARIA DEL TRIUNFO  NO CORRESPONDE   

  _COD_CARRETERA_NORM  
0              LM-671  
1       NOCORRESPONDE  
2             LI-1150  
3               PE-1N  
4       NOCORRESPONDE  


## 2. Fuente MTC: Red Vial Nacional 2022-2025

**Relaci?n con el problema:** las caracter?sticas de la infraestructura vial pueden influir en la severidad del siniestro. Esta fuente permite agregar informaci?n oficial del SINAC, como superficie, estado, n?mero de carriles, concesi?n, corredor log?stico y longitud de ruta.

**Llave de cruce:** `COD CARRETERA` del ONSV contra `CODIGO_RUTA` del MTC. La cobertura no ser? 100% porque el dataset del ONSV incluye v?as urbanas, provinciales, rutas sin clasificar y registros con `NO CORRESPONDE`.

In [101]:
mtc = pd.read_csv(PATHS["mtc_rvn"], encoding="latin1", sep=None, engine="python")
print(f"MTC Red Vial Nacional: {mtc.shape[0]:,} filas x {mtc.shape[1]} columnas")
print(mtc.columns.tolist())
mtc.head(3)

MTC Red Vial Nacional: 34,761 filas x 21 columnas
['ID_RVN', 'ID_DEPARTAMENTO', 'DEPARTAMENTO', 'CODIGO_RUTA', 'TRAYECTORIA', 'INICIO', 'FINAL', 'CLASIFICACION_EJE', 'JERARQUIA', 'LONGITUD', 'NRO_CARRILES', 'SUPERFICIE', 'SUPERFICIE_L', 'ESTADO', 'ESTADO_L', 'CODIGO_CONCESION', 'NOMBRE_CONCESION', 'CODIGO_LOGISTICO', 'CORREDOR_LOGISTICO', 'FECHA_CORTE', 'ES_CONCES']


,ID_RVN,ID_DEPARTAMENTO,DEPARTAMENTO,CODIGO_RUTA,TRAYECTORIA,INICIO,FINAL,CLASIFICACION_EJE,JERARQUIA,LONGITUD,...,SUPERFICIE,SUPERFICIE_L,ESTADO,ESTADO_L,CODIGO_CONCESION,NOMBRE_CONCESION,CODIGO_LOGISTICO,CORREDOR_LOGISTICO,FECHA_CORTE,ES_CONCES
0,1,20,PIURA,PE-02,Emp. PE-1N (Dv. Paita) - Paita,0.000,7.918,Transversal,RN,7.918,...,1,Pavimentado,1,Bueno,2,Eje Multimodal Amazonas Norte,C02,CO 2: Paita - Piura- Dv. Olmos,20221231,1
1,2,20,PIURA,PE-02,Emp. PE-1N (Dv. Paita) - Paita,7.918,40.095,Transversal,RN,32.177,...,1,Pavimentado,1,Bueno,2,Eje Multimodal Amazonas Norte,C02,CO 2: Paita - Piura- Dv. Olmos,20221231,1
2,3,20,PIURA,PE-02,Emp. PE-1N (Dv. Paita) - Paita,40.095,49.058,Transversal,RN,8.963,...,1,Pavimentado,1,Bueno,2,Eje Multimodal Amazonas Norte,C02,CO 2: Paita - Piura- Dv. Olmos,20221231,1


In [102]:
mtc = mtc.copy()
mtc["_COD_RUTA_NORM"] = mtc["CODIGO_RUTA"].apply(normalizar_codigo_ruta)
mtc["LONGITUD"] = pd.to_numeric(mtc["LONGITUD"], errors="coerce")
mtc["NRO_CARRILES"] = pd.to_numeric(mtc["NRO_CARRILES"], errors="coerce")
mtc["ES_CONCES"] = pd.to_numeric(mtc["ES_CONCES"], errors="coerce")

mtc_agg = (
    mtc.groupby("_COD_RUTA_NORM", dropna=False)
    .apply(lambda g: pd.Series({
        "MTC_RVN_LONGITUD_KM": g["LONGITUD"].sum(min_count=1),
        "MTC_RVN_N_CARRILES_MEDIANA": g["NRO_CARRILES"].median(),
        "MTC_RVN_SUPERFICIE_DOMINANTE": moda_ponderada_por_longitud(g, "SUPERFICIE_L"),
        "MTC_RVN_ESTADO_DOMINANTE": moda_ponderada_por_longitud(g, "ESTADO_L"),
        "MTC_RVN_ES_CONCESIONADA": int((g["ES_CONCES"].fillna(0) > 0).any()),
        "MTC_RVN_CONCESION_PRINCIPAL": moda_ponderada_por_longitud(g, "NOMBRE_CONCESION"),
        "MTC_RVN_CORREDOR_LOGISTICO": moda_ponderada_por_longitud(g, "CORREDOR_LOGISTICO"),
        "MTC_RVN_N_TRAMOS": len(g),
    }))
    .reset_index()
)

# Evitar strings vac?os como dato v?lido en concesi?n/corredor.
for col in ["MTC_RVN_CONCESION_PRINCIPAL", "MTC_RVN_CORREDOR_LOGISTICO"]:
    mtc_agg[col] = mtc_agg[col].replace({"": np.nan, " ": np.nan})

print(f"Rutas MTC agregadas: {mtc_agg.shape[0]:,}")
mtc_agg.head()

Rutas MTC agregadas: 160


,_COD_RUTA_NORM,MTC_RVN_LONGITUD_KM,MTC_RVN_N_CARRILES_MEDIANA,MTC_RVN_SUPERFICIE_DOMINANTE,MTC_RVN_ESTADO_DOMINANTE,MTC_RVN_ES_CONCESIONADA,MTC_RVN_CONCESION_PRINCIPAL,MTC_RVN_CORREDOR_LOGISTICO,MTC_RVN_N_TRAMOS
0,PE-02,245.290,2.0,Pavimentado,Bueno,1,Eje Multimodal Amazonas Norte,Corredor IIRSA Norte,18
1,PE-02A,722.900,2.0,Asfaltado económico,Bueno,1,Emp.PE-1B-Buenos Aires - Canchaque,No corresponde,123
2,PE-02B,642.004,1.0,Asfaltado económico,Regular,0,No concesionado,No corresponde,108
3,PE-02C,35.985,-1.0,Pavimentado,Regular,0,No concesionado,No corresponde,25
4,PE-02E,22.413,-1.0,Afirmado,Regular,0,No concesionado,No corresponde,15


In [103]:
cols_mtc_rvn = [c for c in mtc_agg.columns if c.startswith("MTC_RVN_")]
df_antes = df.copy()
df = df.merge(mtc_agg, left_on="_COD_CARRETERA_NORM", right_on="_COD_RUTA_NORM", how="left")
df = df.drop(columns=["_COD_RUTA_NORM"], errors="ignore")

cov_mtc_rvn, cruz_mtc_rvn, no_mtc_rvn = validar_merge(df_antes, df, "MTC Red Vial Nacional 2022-2025", cols_mtc_rvn)

print("\nEjemplos sin cruce MTC RVN:")
df.loc[df[cols_mtc_rvn].notna().any(axis=1) == False, ["COD CARRETERA", "RED VIAL", "DEPARTAMENTO"]].drop_duplicates().head(10)

VALIDACI?N POST-MERGE: MTC Red Vial Nacional 2022-2025
Filas antes:   9,106
Filas despu?s: 9,106
Variable objetivo: inalterada
Cobertura fuente: 55.04% (5,012 cruzados; 4,094 no cruzados)
Columnas nuevas: ['MTC_RVN_LONGITUD_KM', 'MTC_RVN_N_CARRILES_MEDIANA', 'MTC_RVN_SUPERFICIE_DOMINANTE', 'MTC_RVN_ESTADO_DOMINANTE', 'MTC_RVN_ES_CONCESIONADA', 'MTC_RVN_CONCESION_PRINCIPAL', 'MTC_RVN_CORREDOR_LOGISTICO', 'MTC_RVN_N_TRAMOS']

Ejemplos sin cruce MTC RVN:


,COD CARRETERA,RED VIAL,DEPARTAMENTO
0,LM-671,PROVINCIAL,LIMA
1,NO CORRESPONDE,URBANO,LIMA
2,LI-1150,PROVINCIAL,LA LIBERTAD
5,NO CORRESPONDE,URBANO,SAN MARTIN
7,NO CORRESPONDE,URBANO,PIURA
10,NO CORRESPONDE,URBANO,ANCASH
11,PI-102,DEPARTAMENTAL,PIURA
18,PI-897,PROVINCIAL,PIURA
21,SIN CLASIFICAR,SIN CLASIFICAR,CUSCO
22,PU-152,DEPARTAMENTAL,PUNO


## 3. Fuente IDH PNUD/IPE 2019

**Relaci?n con el problema:** el IDH y sus componentes funcionan como variables de contexto territorial: desarrollo humano, educaci?n, esperanza de vida e ingreso pueden aproximar condiciones de acceso a servicios, respuesta institucional y vulnerabilidad social.

**Llave de cruce:** se prioriza `DEPARTAMENTO + PROVINCIA + DISTRITO`, usando el archivo oficial del IPE/PNUD. El IDH 2019 es previo al periodo de siniestros 2021-2025, por lo que se interpreta como indicador estructural del territorio.

In [104]:
idh_raw = pd.read_excel(PATHS["idh"], sheet_name="IDH 2019", header=None)
print(f"IDH raw: {idh_raw.shape[0]:,} filas x {idh_raw.shape[1]} columnas")
idh_raw.head(15)

IDH raw: 2,305 filas x 17 columnas


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,índice de Desarrollo Humano 2019,NaN,NaN,NaN,NaN,NaN,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Valores normalizados,NaN,NaN,NaN,NaN,Índice de desarrollo Humano (IDH)
3,UBIGEO,DEPARTAMENTO,NaN,NaN,NaN,Población,Esperanza de vida al nacer,Población (18 años) con Educ. secundaria completa,Años de educación (Poblac. 25 y más),Ingreso familiar per cápita,NaN,Esperanza de vida al nacer,Población (18 años) con Educ. secundaria completa,Años de educación (Poblac. 25 y más),Logro educativo,Ingreso familiar per cápita,NaN
4,NaN,NaN,Provincia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Distrito,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,000000,PERÚ,NaN,NaN,NaN,31296142.413458,75.422882,67.665801,9.135085,1032.160379,NaN,0.840381,0.676658,0.516555,0.591212,0.404528,0.585764
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:
# El archivo IDH 2019 tiene jerarqu?a: departamento, provincia y distrito en filas sucesivas.
# A partir de la fila 8 est?n los datos. Las columnas 0,1,2 contienen UBIGEO / nivel / nombre.
idh = idh_raw.iloc[8:, [0, 1, 2, 5, 6, 7, 8, 9, 16]].copy()
idh.columns = [
    "UBIGEO", "NIVEL_O_DEP", "NOMBRE", "IDH_POBLACION_2019", "IDH_ESP_VIDA_2019",
    "IDH_EDUC_SECUND_PCT_2019", "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_2019"
]
idh = idh.dropna(subset=["UBIGEO"]).copy()
idh["UBIGEO"] = pd.to_numeric(idh["UBIGEO"], errors="coerce").astype("Int64").astype(str).str.zfill(6)

for col in ["IDH_POBLACION_2019", "IDH_ESP_VIDA_2019", "IDH_EDUC_SECUND_PCT_2019", "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_2019"]:
    idh[col] = pd.to_numeric(idh[col], errors="coerce")

idh["TIPO_NIVEL"] = np.select(
    [idh["UBIGEO"].str.endswith("0000"), idh["UBIGEO"].str.endswith("00")],
    ["DEPARTAMENTO", "PROVINCIA"],
    default="DISTRITO"
)

idh["DEPARTAMENTO_IDH"] = np.where(idh["TIPO_NIVEL"].eq("DEPARTAMENTO"), idh["NIVEL_O_DEP"], np.nan)
idh["DEPARTAMENTO_IDH"] = idh["DEPARTAMENTO_IDH"].ffill()
idh["PROVINCIA_IDH"] = np.where(idh["TIPO_NIVEL"].eq("PROVINCIA"), idh["NOMBRE"], np.nan)
idh["PROVINCIA_IDH"] = idh["PROVINCIA_IDH"].ffill()
idh["DISTRITO_IDH"] = np.where(idh["TIPO_NIVEL"].eq("DISTRITO"), idh["NOMBRE"], np.nan)

idh_dist = idh[idh["TIPO_NIVEL"].eq("DISTRITO")].copy()
idh_dist["_DEP_NORM"] = idh_dist["DEPARTAMENTO_IDH"].apply(normalizar_texto)
idh_dist["_PROV_NORM"] = idh_dist["PROVINCIA_IDH"].apply(normalizar_texto)
idh_dist["_DIST_NORM"] = idh_dist["DISTRITO_IDH"].apply(normalizar_texto)

cols_idh = [
    "IDH_2019", "IDH_ESP_VIDA_2019", "IDH_EDUC_SECUND_PCT_2019",
    "IDH_ANIOS_EDUC_2019", "IDH_INGRESO_PERCAP_2019", "IDH_POBLACION_2019"
]
idh_dist_sel = idh_dist[["_DEP_NORM", "_PROV_NORM", "_DIST_NORM", "UBIGEO"] + cols_idh].drop_duplicates(["_DEP_NORM", "_PROV_NORM", "_DIST_NORM"])
idh_dist_sel = idh_dist_sel.rename(columns={"UBIGEO": "IDH_UBIGEO"})

print(f"Distritos IDH 2019 disponibles: {idh_dist_sel.shape[0]:,}")
idh_dist_sel.head()

Distritos IDH 2019 disponibles: 1,876


,_DEP_NORM,_PROV_NORM,_DIST_NORM,IDH_UBIGEO,IDH_2019,IDH_ESP_VIDA_2019,IDH_EDUC_SECUND_PCT_2019,IDH_ANIOS_EDUC_2019,IDH_INGRESO_PERCAP_2019,IDH_POBLACION_2019
12,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,010101,0.642361,72.192674,79.562705,10.018339,1259.130244,33038.252677
13,AMAZONAS,CHACHAPOYAS,ASUNCION,010102,0.423032,71.364553,71.006832,6.013571,561.102634,267.447537
14,AMAZONAS,CHACHAPOYAS,BALSAS,010103,0.315308,68.594405,32.772384,5.193542,415.021047,1467.034261
15,AMAZONAS,CHACHAPOYAS,CHETO,010104,0.345746,77.528252,52.210906,4.584966,398.663339,584.745182
16,AMAZONAS,CHACHAPOYAS,CHILIQUIN,010105,0.275038,72.460151,36.868932,3.712853,325.930531,390.717345


In [106]:
df_antes = df.copy()
df = df.merge(idh_dist_sel, on=["_DEP_NORM", "_PROV_NORM", "_DIST_NORM"], how="left")

cov_idh, cruz_idh, no_idh = validar_merge(df_antes, df, "IDH PNUD/IPE 2019 distrital", cols_idh)

print("\nEjemplos sin cruce IDH:")
df.loc[df["IDH_2019"].isna(), ["DEPARTAMENTO", "PROVINCIA", "DISTRITO"]].drop_duplicates().head(15)

VALIDACI?N POST-MERGE: IDH PNUD/IPE 2019 distrital
Filas antes:   9,106
Filas despu?s: 9,106
Variable objetivo: inalterada
Cobertura fuente: 96.91% (8,825 cruzados; 281 no cruzados)
Columnas nuevas: ['IDH_2019', 'IDH_ESP_VIDA_2019', 'IDH_EDUC_SECUND_PCT_2019', 'IDH_ANIOS_EDUC_2019', 'IDH_INGRESO_PERCAP_2019', 'IDH_POBLACION_2019']

Ejemplos sin cruce IDH:


,DEPARTAMENTO,PROVINCIA,DISTRITO
53,ICA,NASCA,MARCONA
75,PASCO,DANIEL ALCIDES CARRION,TAPUC
80,CALLAO,CALLAO,LA PERLA
85,CALLAO,CALLAO,CALLAO
109,TACNA,TACNA,CORONEL GREGORIO ALBARRACIN LANCHIPA
122,CALLAO,CALLAO,VENTANILLA
166,HUANUCO,LEONCIO PRADO,SANTO DOMINGO DE ANDA
247,ICA,NASCA,VISTA ALEGRE
274,HUANUCO,HUANUCO,QUISQUI (KICHKI)
404,LIMA,LIMA,PUEBLO LIBRE


## 4. Fuente MTC: infraestructura vial por departamento 2025

**Relaci?n con el problema:** esta fuente agrega el contexto vial departamental: longitud total de red, red nacional/departamental/vecinal y proporci?n pavimentada. Es una variable de exposici?n e infraestructura a nivel territorial.

**Llave de cruce:** `DEPARTAMENTO`. Se usa la hoja 2025 porque es el corte m?s reciente disponible en el archivo descargado del MTC.

In [107]:
mtc_dep_raw = pd.read_excel(PATHS["mtc_departamento"], sheet_name="2025", header=None)
print(f"MTC infraestructura departamental raw: {mtc_dep_raw.shape[0]:,} filas x {mtc_dep_raw.shape[1]} columnas")
mtc_dep_raw.head(20)

MTC infraestructura departamental raw: 39 filas x 15 columnas


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,"Infraestructura Vial Existente del SINAC, seg...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,(Clasificador de Rutas D.S.011-2016-MTC al 31 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Kilómetros,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,DEPARTAMENTO,LONGITUD \nTOTAL,NaN,NACIONAL,NaN,NaN,NaN,DEPARTAMENTAL 1/,NaN,NaN,NaN,VECINAL 2/,NaN,NaN
7,NaN,NaN,NaN,NaN,SUB TOTAL,Pavimentada,No Pavimentada,NaN,SUB-TOTAL,Pavimento,No Pavimentada,NaN,SUB-TOTAL,Pavimento,No Pavimentada
8,NaN,TOTAL,176936.349875,NaN,27679.789,22875.143,4804.646,NaN,27828.458,7700.215,20128.243,NaN,121428.102875,3944.96,117483.142875
9,NaN,Amazonas,3903.732,NaN,1155.121,849.877,305.244,NaN,585.018,51.15,533.868,NaN,2163.593,42.564,2121.029


In [108]:
# En este Excel, la tabla empieza en la columna B y tiene columnas vac?as separadoras.
# Se seleccionan: departamento, total, nacional total/pav/no pav, departamental total/pav/no pav, vecinal total/pav/no pav.
cols_excel = [1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]
mtc_dep = mtc_dep_raw.iloc[9:, cols_excel].copy()
mtc_dep.columns = [
    "DEPARTAMENTO_MTC_DEP", "MTC_DEP_RED_TOTAL_KM", "MTC_DEP_NACIONAL_TOTAL_KM",
    "MTC_DEP_NACIONAL_PAV_KM", "MTC_DEP_NACIONAL_NO_PAV_KM",
    "MTC_DEP_DEPARTAMENTAL_TOTAL_KM", "MTC_DEP_DEPARTAMENTAL_PAV_KM", "MTC_DEP_DEPARTAMENTAL_NO_PAV_KM",
    "MTC_DEP_VECINAL_TOTAL_KM", "MTC_DEP_VECINAL_PAV_KM", "MTC_DEP_VECINAL_NO_PAV_KM",
]
mtc_dep = mtc_dep.dropna(subset=["DEPARTAMENTO_MTC_DEP"]).copy()
mtc_dep = mtc_dep[~mtc_dep["DEPARTAMENTO_MTC_DEP"].astype(str).str.upper().str.contains("FUENTE|ELABORACION|NOTA", na=False)].copy()
mtc_dep["_DEP_NORM"] = mtc_dep["DEPARTAMENTO_MTC_DEP"].apply(normalizar_texto)

cols_mtc_dep = [c for c in mtc_dep.columns if c.startswith("MTC_DEP_")]
for col in cols_mtc_dep:
    mtc_dep[col] = pd.to_numeric(mtc_dep[col], errors="coerce")

mtc_dep["MTC_DEP_PAV_TOTAL_KM"] = mtc_dep["MTC_DEP_NACIONAL_PAV_KM"] + mtc_dep["MTC_DEP_DEPARTAMENTAL_PAV_KM"] + mtc_dep["MTC_DEP_VECINAL_PAV_KM"]
mtc_dep["MTC_DEP_NO_PAV_TOTAL_KM"] = mtc_dep["MTC_DEP_NACIONAL_NO_PAV_KM"] + mtc_dep["MTC_DEP_DEPARTAMENTAL_NO_PAV_KM"] + mtc_dep["MTC_DEP_VECINAL_NO_PAV_KM"]
mtc_dep["MTC_DEP_PCT_PAVIMENTADA"] = np.where(
    mtc_dep["MTC_DEP_RED_TOTAL_KM"] > 0,
    mtc_dep["MTC_DEP_PAV_TOTAL_KM"] / mtc_dep["MTC_DEP_RED_TOTAL_KM"] * 100,
    np.nan,
)
cols_mtc_dep_final = [
    "MTC_DEP_RED_TOTAL_KM", "MTC_DEP_NACIONAL_TOTAL_KM", "MTC_DEP_DEPARTAMENTAL_TOTAL_KM",
    "MTC_DEP_VECINAL_TOTAL_KM", "MTC_DEP_PAV_TOTAL_KM", "MTC_DEP_NO_PAV_TOTAL_KM",
    "MTC_DEP_PCT_PAVIMENTADA"
]
mtc_dep_sel = mtc_dep[["_DEP_NORM"] + cols_mtc_dep_final].drop_duplicates("_DEP_NORM")

print(f"Departamentos MTC 2025: {mtc_dep_sel.shape[0]}")
mtc_dep_sel.head()

Departamentos MTC 2025: 27


,_DEP_NORM,MTC_DEP_RED_TOTAL_KM,MTC_DEP_NACIONAL_TOTAL_KM,MTC_DEP_DEPARTAMENTAL_TOTAL_KM,MTC_DEP_VECINAL_TOTAL_KM,MTC_DEP_PAV_TOTAL_KM,MTC_DEP_NO_PAV_TOTAL_KM,MTC_DEP_PCT_PAVIMENTADA
9,AMAZONAS,3903.732,1155.121,585.018,2163.593,943.591,2960.141,24.171511
10,ANCASH,10767.648,1885.063,1222.721,7659.864,2100.343,8667.305,19.506052
11,APURIMAC,8358.971,1282.451,1298.097,5778.423,1432.683,6926.288,17.139466
12,AREQUIPA,10322.272,1495.236,1749.739,7077.297,2886.595,7435.677,27.964725
13,AYACUCHO,12678.733,1805.074,1774.034,9099.625,2485.375,10193.358,19.602708


In [109]:
df_antes = df.copy()
df = df.merge(mtc_dep_sel, on="_DEP_NORM", how="left")

cov_mtc_dep, cruz_mtc_dep, no_mtc_dep = validar_merge(df_antes, df, "MTC infraestructura vial por departamento 2025", cols_mtc_dep_final)

print("\nEjemplos sin cruce MTC departamento:")
df.loc[df["MTC_DEP_RED_TOTAL_KM"].isna(), ["DEPARTAMENTO"]].drop_duplicates().head(10)

VALIDACI?N POST-MERGE: MTC infraestructura vial por departamento 2025
Filas antes:   9,106
Filas despu?s: 9,106
Variable objetivo: inalterada
Cobertura fuente: 100.00% (9,106 cruzados; 0 no cruzados)
Columnas nuevas: ['MTC_DEP_RED_TOTAL_KM', 'MTC_DEP_NACIONAL_TOTAL_KM', 'MTC_DEP_DEPARTAMENTAL_TOTAL_KM', 'MTC_DEP_VECINAL_TOTAL_KM', 'MTC_DEP_PAV_TOTAL_KM', 'MTC_DEP_NO_PAV_TOTAL_KM', 'MTC_DEP_PCT_PAVIMENTADA']

Ejemplos sin cruce MTC departamento:


,DEPARTAMENTO


## 5. Fuentes evaluadas y no integradas en esta versi?n

- **INEI / EstaDist:** se evalu? para indicadores distritales, pero el portal requiere selecci?n manual de distritos para exportaci?n. Se deja como mejora futura para no introducir un proceso manual incompleto.
- **SENAMHI:** se evalu? para clima objetivo, pero requiere descarga por estaci?n y cruce espaciotemporal por coordenadas y fecha exacta. El dataset limpio conserva a?o/mes/hora, pero no la fecha completa original.
- **MTC flujo vehicular por peajes:** ?til como proxy de exposici?n vehicular, pero su cruce correcto requiere peajes georreferenciados o agregaci?n por ruta/departamento. Se deja como mejora futura.

## 6. Trazabilidad de variables externas

Se documenta cada variable nueva con fuente, archivo origen, URL, llave de cruce, nivel geogr?fico, cobertura y observaciones.

In [110]:
trazabilidad = []

agregar_trazabilidad(
    trazabilidad, cols_mtc_rvn,
    fuente="MTC - Red Vial Nacional SINAC 2022-2025",
    archivo_origen=PATHS["mtc_rvn"].as_posix(),
    url="https://www.datosabiertos.gob.pe/dataset/red-vial-nacional-del-sistema-nacional-de-carreteras-2022-2025-ministerio-de-transportes-y-comunicaciones-mtc",
    llave_cruce="COD CARRETERA -> CODIGO_RUTA",
    anio_dato="2022-2025",
    nivel_geografico="Ruta / tramo agregado por c?digo de ruta",
    cobertura=cov_mtc_rvn,
    cruzados=cruz_mtc_rvn,
    no_cruzados=no_mtc_rvn,
    observaciones="Cobertura parcial esperada: no aplica a v?as urbanas, NO CORRESPONDE, SIN CLASIFICAR o rutas no nacionales."
)

agregar_trazabilidad(
    trazabilidad, cols_idh + ["IDH_UBIGEO"],
    fuente="PNUD/IPE - IDH y componentes 2019",
    archivo_origen=PATHS["idh"].as_posix(),
    url="https://ipe.org.pe/indice-de-desarrollo-humano-idh/",
    llave_cruce="DEPARTAMENTO + PROVINCIA + DISTRITO",
    anio_dato="2019",
    nivel_geografico="Distrito",
    cobertura=cov_idh,
    cruzados=cruz_idh,
    no_cruzados=no_idh,
    observaciones="Indicador estructural previo al periodo 2021-2025; puede haber no cruces por diferencias de nomenclatura distrital."
)

agregar_trazabilidad(
    trazabilidad, cols_mtc_dep_final,
    fuente="MTC - Infraestructura vial existente del SINAC por departamento",
    archivo_origen=PATHS["mtc_departamento"].as_posix(),
    url="https://www.gob.pe/institucion/mtc/informes-publicaciones/344790-estadistica-infraestructura-de-transportes-infraestructura-vial",
    llave_cruce="DEPARTAMENTO",
    anio_dato="2025 (junio)",
    nivel_geografico="Departamento",
    cobertura=cov_mtc_dep,
    cruzados=cruz_mtc_dep,
    no_cruzados=no_mtc_dep,
    observaciones="Variables agregadas a nivel departamental; no capturan heterogeneidad intradepartamental."
)

df_trazabilidad = pd.DataFrame(trazabilidad)
df_trazabilidad

,variable,fuente,archivo_origen,url_fuente,llave_cruce,anio_dato,nivel_geografico,cobertura_pct,registros_cruzados,registros_no_cruzados,observaciones
0,MTC_RVN_LONGITUD_KM,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
1,MTC_RVN_N_CARRILES_MEDIANA,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
2,MTC_RVN_SUPERFICIE_DOMINANTE,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
3,MTC_RVN_ESTADO_DOMINANTE,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
4,MTC_RVN_ES_CONCESIONADA,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
5,MTC_RVN_CONCESION_PRINCIPAL,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
6,MTC_RVN_CORREDOR_LOGISTICO,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
7,MTC_RVN_N_TRAMOS,MTC - Red Vial Nacional SINAC 2022-2025,/home/node-vault07/projects/siniestros-fatales...,https://www.datosabiertos.gob.pe/dataset/red-v...,COD CARRETERA -> CODIGO_RUTA,2022-2025,Ruta / tramo agregado por c?digo de ruta,55.04,5012,4094,Cobertura parcial esperada: no aplica a v?as u...
8,IDH_2019,PNUD/IPE - IDH y componentes 2019,/home/node-vault07/projects/siniestros-fatales...,https://ipe.org.pe/indice-de-desarrollo-humano...,DEPARTAMENTO + PROVINCIA + DISTRITO,2019,Distrito,96.91,8825,281,Indicador estructural previo al periodo 2021-2...
9,IDH_ESP_VIDA_2019,PNUD/IPE - IDH y componentes 2019,/home/node-vault07/projects/siniestros-fatales...,https://ipe.org.pe/indice-de-desarrollo-humano...,DEPARTAMENTO + PROVINCIA + DISTRITO,2019,Distrito,96.91,8825,281,Indicador estructural previo al periodo 2021-2...


## 7. Validaci?n global y exportaci?n

Antes de exportar se verifica que no se hayan duplicado filas ni alterado la variable objetivo. Las columnas auxiliares de normalizaci?n se eliminan del CSV final.

In [111]:
cols_aux = [c for c in df.columns if c.startswith("_")]
df_final = df.drop(columns=cols_aux, errors="ignore").copy()

assert len(df_final) == filas_originales, "La cantidad de filas cambi? durante el enriquecimiento."
assert df_final["CATEGORIA_SEVERIDAD"].value_counts().sort_index().to_dict() == dist_objetivo_original, "La variable objetivo cambi?."

cols_originales = list(df_original.columns)
cols_nuevas = [c for c in df_final.columns if c not in cols_originales]

print("=" * 72)
print("VALIDACI?N GLOBAL FINAL")
print("=" * 72)
print(f"Filas finales: {len(df_final):,}")
print(f"Columnas originales: {len(cols_originales)}")
print(f"Columnas nuevas: {len(cols_nuevas)}")
print(f"Columnas finales: {df_final.shape[1]}")
print("\nVariables nuevas:")
for col in cols_nuevas:
    print(f"- {col}: {df_final[col].notna().mean() * 100:.2f}% cobertura")

# Exportar con utf-8-sig para que Excel reconozca tildes y ?.
df_final.to_csv(PATHS["salida_enriquecido"], index=False, encoding="utf-8-sig")
df_trazabilidad.to_csv(PATHS["salida_trazabilidad"], index=False, encoding="utf-8-sig")

print("\nArchivos generados:")
print(PATHS["salida_enriquecido"])
print(PATHS["salida_trazabilidad"])
print("\nPr?ximo paso: usar data/procesada/siniestros_enriquecido.csv como entrada del notebook 03.")

VALIDACI?N GLOBAL FINAL
Filas finales: 9,106
Columnas originales: 36
Columnas nuevas: 22
Columnas finales: 58

Variables nuevas:
- MTC_RVN_LONGITUD_KM: 55.04% cobertura
- MTC_RVN_N_CARRILES_MEDIANA: 55.04% cobertura
- MTC_RVN_SUPERFICIE_DOMINANTE: 55.04% cobertura
- MTC_RVN_ESTADO_DOMINANTE: 55.04% cobertura
- MTC_RVN_ES_CONCESIONADA: 55.04% cobertura
- MTC_RVN_CONCESION_PRINCIPAL: 55.04% cobertura
- MTC_RVN_CORREDOR_LOGISTICO: 55.04% cobertura
- MTC_RVN_N_TRAMOS: 55.04% cobertura
- IDH_UBIGEO: 96.91% cobertura
- IDH_2019: 96.91% cobertura
- IDH_ESP_VIDA_2019: 96.91% cobertura
- IDH_EDUC_SECUND_PCT_2019: 96.91% cobertura
- IDH_ANIOS_EDUC_2019: 96.91% cobertura
- IDH_INGRESO_PERCAP_2019: 96.91% cobertura
- IDH_POBLACION_2019: 96.91% cobertura
- MTC_DEP_RED_TOTAL_KM: 100.00% cobertura
- MTC_DEP_NACIONAL_TOTAL_KM: 100.00% cobertura
- MTC_DEP_DEPARTAMENTAL_TOTAL_KM: 100.00% cobertura
- MTC_DEP_VECINAL_TOTAL_KM: 100.00% cobertura
- MTC_DEP_PAV_TOTAL_KM: 100.00% cobertura
- MTC_DEP_NO_PAV_TO

## 8. Resumen metodol?gico

El dataset enriquecido conserva todos los registros originales y agrega variables externas reales. La cobertura de MTC Red Vial Nacional es parcial por la propia naturaleza de la fuente: solo representa rutas nacionales del SINAC y no cubre v?as urbanas o registros sin c?digo de carretera. El IDH se integra a nivel distrital cuando los nombres coinciden tras normalizaci?n, mientras que la infraestructura vial departamental entrega contexto territorial agregado con cobertura esperada cercana al 100%.